# Benchmark: EN→CY Translation

Evaluates our model against the Opus-MT baseline using BLEU and chrF on the Cardiff University test split.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import sacrebleu
import yaml
from transformers import MarianMTModel, MarianTokenizer

from src import datasets, utils
from src.hf_wrapper import EnCyForTranslation
from src.tokenizer import load_tokenizer

CONFIG = "base"
NUM_SENTENCES = 1000
BATCH_SIZE = 32

with open(f"../configs/{CONFIG}.yaml") as f:
    config = yaml.safe_load(f)

run_path = utils.get_run_path(config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load test data

In [ ]:
ds = datasets.get_dataset("train", config)
test_ds = ds["test"].select(range(min(NUM_SENTENCES, len(ds["test"]))))
src_sentences = test_ds["text_en"]
ref_sentences = test_ds["text_cy"]
print(f"Test sentences: {len(src_sentences)}")

## Translation helpers

In [ ]:
def translate_ours(model, tokenizer, sentences, max_length):
    model.eval()
    results = []
    for i in range(0, len(sentences), BATCH_SIZE):
        batch = sentences[i : i + BATCH_SIZE]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_length=max_length, num_beams=4)
        results.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
    return results


def translate_opus(model, tokenizer, sentences):
    results = []
    for i in range(0, len(sentences), BATCH_SIZE):
        batch = sentences[i : i + BATCH_SIZE]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            output_ids = model.generate(**inputs)
        results.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
    return results


def score(hypotheses, references):
    bleu = sacrebleu.corpus_bleu(hypotheses, [references], tokenize="intl")
    chrf = sacrebleu.corpus_chrf(hypotheses, [references])
    return {"BLEU": round(bleu.score, 2), "chrF": round(chrf.score, 2)}

## Our model

In [ ]:
our_model = EnCyForTranslation.from_run(run_path).to(device)
our_tokenizer = load_tokenizer(run_path)

our_hyps = translate_ours(our_model, our_tokenizer, src_sentences, config["model"]["max_length"])
our_scores = score(our_hyps, ref_sentences)
print("Our model:", our_scores)

del our_model
torch.cuda.empty_cache()

## Opus-MT baseline

In [ ]:
opus_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-cy")
opus_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-cy").to(device)

opus_hyps = translate_opus(opus_model, opus_tokenizer, src_sentences)
opus_scores = score(opus_hyps, ref_sentences)
print("Opus-MT:", opus_scores)

del opus_model
torch.cuda.empty_cache()

## Results

In [ ]:
import pandas as pd

results = {
    "Our model": our_scores,
    "Opus-MT": opus_scores,
}
df = pd.DataFrame(results).T
display(df)

## Example translations

In [ ]:
N = 10
for i in range(N):
    print(f"EN:        {src_sentences[i]}")
    print(f"Reference: {ref_sentences[i]}")
    print(f"Ours:      {our_hyps[i]}")
    print(f"Opus-MT:   {opus_hyps[i]}")
    print()